# 🗣️ 10.18 Spark SQL

Welcome to **SECTION F: ANALYTICAL PROCESSING**!

Up until now, we have been using the PySpark DataFrame API (Python code with methods like `.select()`, `.filter()`, and `.groupBy()`). 

But what if you love traditional SQL? What if you are collaborating with a team of Data Analysts who only know SQL? 

Spark has an incredible feature: It allows you to write raw, standard SQL queries and execute them directly against your distributed Petabyte-scale DataFrames. In this lesson, we will learn how to turn our Python objects into "Virtual Tables" and query them using the **Spark SQL Engine**.

### 🎯 Learning Objectives
* Understand how to bridge PySpark and SQL using **Temporary Views**.
* Write and execute raw **SQL Queries** using `spark.sql()`.
* Discover how the **Spark SQL Engine** treats Python API code and raw SQL strings as the exact same thing under the hood.
* See how enabling SQL on Big Data changes the dynamics of modern data teams.

## 1. The Bridge: Temporary Views

A standard SQL engine cannot query a Python variable (like `df`). SQL engines only know how to query "Tables".

To bridge this gap, Spark uses **Temporary Views**. 

A Temporary View is a virtual table. It doesn't physically copy or move your data; it simply gives your Python DataFrame a recognizable "SQL Table Name" so the Spark SQL engine can find it.

<img src="../assets/Module_10/10_18_01.png" alt="A visual of a bridge connecting two islands. The left island is labeled 'Python DataFrames' and the right island is labeled 'Traditional SQL'. The bridge connecting them is labeled 'createOrReplaceTempView()'." width="75%">

In [ ]:
# 0. Setup: Let's start our SparkSession and create some data.
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spark_SQL_Demo").master("local[*]").getOrCreate()

# Create an Employees DataFrame
emp_data = [
    (1, "Alice", "Engineering", 120000),
    (2, "Bob", "Sales", 85000),
    (3, "Charlie", "Engineering", 105000),
    (4, "Diana", "Marketing", 90000)
]
emp_df = spark.createDataFrame(emp_data, ["id", "name", "dept", "salary"])

print("1. We have a Python DataFrame named 'emp_df'. SQL cannot see this.")
emp_df.show()

# --- CREATING THE TEMPORARY VIEW ---
print("2. Creating a Temporary View named 'employees_table'...")

emp_df.createOrReplaceTempView("employees_table")

print("Success! The Spark SQL engine can now query 'employees_table'.")

## 2. Executing SQL Queries

Once the Temporary View is created, you can use the `spark.sql("...")` function to write standard SQL.

The best part? Because it is standard ANSI SQL, you can use all your favorite SQL commands: `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY`, and even `JOIN`!

In [ ]:
print("--- BASIC SQL QUERY ---")
# Let's find all Engineers making over $100k

sql_query = """
SELECT name, salary 
FROM employees_table 
WHERE dept = 'Engineering' AND salary > 100000
"""

# Execute the query. Notice that spark.sql() returns a standard DataFrame!
results_df = spark.sql(sql_query)
results_df.show()

print("--- SQL AGGREGATION ---")
# Let's calculate the average salary by department
agg_query = """
SELECT dept, AVG(salary) as avg_salary, COUNT(1) as total_employees
FROM employees_table
GROUP BY dept
ORDER BY avg_salary DESC
"""

spark.sql(agg_query).show()

## 3. The Spark SQL Engine (Catalyst Optimizer)

A very common question is: *"Which is faster? Writing PySpark code (`df.filter()`) or writing SQL (`spark.sql()`)?"*

The answer is: **They are exactly the same.**

In Lesson 10.9, we learned about the **Catalyst Optimizer**. The Catalyst Optimizer is the brain of Spark SQL. 

Whether you write a Python command (`df.filter()`) or a raw SQL string (`SELECT * WHERE...`), Spark passes both of them into the exact same Catalyst Optimizer. Catalyst translates both languages into the exact same physical DAG (execution plan).

<img src="../assets/Module_10/10_18_02.png" alt="A funnel labeled 'Catalyst Optimizer' accepting both a SQL query string and PySpark DataFrame API code at the top. The funnel outputs a single, identical optimized DAG execution plan at the bottom." width="75%">

Let's prove this by looking at the execution plans!

In [ ]:
import pyspark.sql.functions as F

print("--- THE PYTHON DATAFRAME WAY ---")
python_df = emp_df.filter(F.col("salary") > 90000).select("name")
python_df.explain()

print("\n--- THE SPARK SQL WAY ---")
sql_df = spark.sql("SELECT name FROM employees_table WHERE salary > 90000")
sql_df.explain()

# If you look closely at the output, the Physical Plan is 100% IDENTICAL.
# (Filter (salary > 90000) -> Project (name))

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

Spark SQL is the ultimate collaboration tool. Here is how it impacts different roles:

| Feature | 🏛️ Data Analyst / DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Tool Preference** | Loves writing pure SQL. Doesn't know Python. | **Comfortable in PySpark (Python) and SQL. Blends them seamlessly.** | Loves Pandas (Python) and Machine Learning libraries. |
| **The Collaboration** | Uses DataBricks or Spark SQL endpoints to write `SELECT` queries against Petabytes of data without needing to learn Python. | **Sets up the architecture. Loads the JSON files, creates the `Temporary Views`, and hands the virtual tables over to the Analysts.** | Uses PySpark for heavy lifting, then extracts a small SQL result set to train models locally. |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Fights over which syntax is "better" (Python vs SQL). Constantly rewrites perfectly good SQL logic into PySpark logic because they think it will run faster.
2. **Mid-Level DE (Your Goal):** Understands that under the Catalyst Optimizer, **Python and SQL are identical in performance**. Uses whichever syntax is cleanest for the specific task. (e.g., using Python for dynamic, looping pipelines, and using SQL for complex, multi-table business logic reporting).
3. **Senior DE:** Implements **Global Temporary Views**. A Senior DE knows that a standard Temp View only lives inside your specific Jupyter notebook session. They implement Global views so an entire company of analysts can query the same live Spark DataFrame simultaneously.

### 🛠️ Where Data Engineering Fits in Modern Organizations
Before Spark SQL, Data Analysts couldn't query massive Data Lakes because they didn't know Java or MapReduce. Data Engineers use Spark SQL to "democratize" the data. By creating Temporary Views on top of Petabyte-scale files, DEs allow anyone who knows basic SQL to analyze Big Data!

## 🔑 Key Takeaways

- **Temporary Views:** Virtual tables created using `createOrReplaceTempView()`. They act as a bridge, allowing the Spark SQL engine to see and query Python DataFrame objects.
- **`spark.sql()`:** The method used to pass a raw SQL string into Spark. It returns a standard Spark DataFrame.
- **No Performance Difference:** Because of the Catalyst Optimizer, a PySpark command (like `.filter()`) and a SQL command (like `WHERE`) compile down into the exact same physical execution plan. Neither is faster than the other.
- **Team Collaboration:** Spark SQL allows Data Engineers to prepare complex datasets using Python, and then expose them as tables so Data Analysts can query them using standard SQL.

## 📝 Knowledge Check Quiz

**Question 1:** If you want to use `spark.sql("SELECT * FROM sales")` on a PySpark DataFrame named `my_df`, what must you do first?
A) Save the DataFrame to a PostgreSQL database.
B) Run `my_df.createOrReplaceTempView("sales")` to register the DataFrame as a virtual table.
C) Convert the DataFrame into an RDD.
D) Write a Java MapReduce job.

**Question 2:** Your coworker claims that you should rewrite all your `spark.sql()` queries into PySpark `.filter()` and `.groupBy()` commands because "Python native methods process data faster than raw SQL strings in Spark." Is your coworker correct?
A) Yes, Python methods skip the parsing phase and execute immediately.
B) No. Both PySpark methods and raw SQL strings are passed into the Catalyst Optimizer, which translates them into the exact same physical execution plan. They have identical performance.
C) Yes, SQL is inherently slow on distributed clusters.
D) No, SQL is actually 100x faster than Python methods.

**Question 3:** What happens to a standard Temporary View when you close your SparkSession (e.g., when you shut down your Jupyter Notebook)?
A) It is permanently saved to the hard drive.
B) It is sent to the Data Analysts.
C) The view disappears, as it is tied exclusively to the lifecycle of that specific SparkSession.
D) The view becomes a Global table.

---

*Answers: 1: B, 2: B, 3: C*

In [ ]:
# Clean up our session
spark.stop()
print("Spark session closed.")

### 🚀 What's Next?
You now have the power of both Python and SQL at your fingertips!

In the next lesson, we are going to dive into one of the most powerful, advanced analytical techniques available in both SQL and PySpark: **10.19 Window Functions**. You will learn how to calculate running totals, moving averages, and ranks across your distributed datasets. Let's go!